# ETL – Exportaciones DANE (Departamento Administrativo Nacional de Estadística de Colombia)

Este notebook implementa un pipeline ETL para **descargar**, **extraer**, **convertir**, **consolidar** y **limpiar** los datos de exportaciones de bienes de Colombia, correspondientes al período **enero de 2021 – octubre de 2025**.  
El flujo de procesamiento sigue el esquema:

**ZIP / XLSX mensuales → CSV intermedios → dataset analítico limpio (Parquet)**.

## Contexto

Las estadísticas oficiales de exportaciones de bienes en Colombia son producidas por el **DANE**, a partir de los **registros administrativos de las Declaraciones de Exportación (DEX)** presentadas por los exportadores ante la **DIAN**, a través del sistema MUISCA.  
Cada DEX constituye la **unidad básica de observación**, y una vez la autoridad aduanera realiza el cierre de la declaración, esta información es validada, depurada y consolidada por el DANE para la producción de las estadísticas oficiales de comerci.{index=0}.

Si bien el DANE publica estas estadísticas con periodicidad **mensual**, la información se encuentra distribuida en **archivos independientes por período (ZIP que contienen excels)**, sin contar con:
- una **API pública**, ni
- un **dataset agregado y listo para análisis longitudinal** que facilite la exploración histórica de las operaciones de exportación a nivel micro (operación) o meso (producto, país, departamento, aduana, modalidad).

Esto dificulta el uso directo de la información para análisis exploratorios, visualización avanzada, modelamiento estadístico o entrenamiento de modeítica predictiva.

## Objetivo del proyecto

El objetivo principal de este proyecto es **diseñar y documentar un proceso reproducible de consolidación y transformación de datos de exportaciones**, que permita:

- Integrar los reportes mensuales publicados por el DANE en un **único dataset histórico coherente**.
- Aplicar reglas de **limpieza, estandarización y control de calidad** sobre variables clave (fechas, valores FOB, pesos, códigos geográficos y arancelarios).
- Preservar los **tipos de datos analíticos** y la trazabilidad de las variables conforme a la metodología oficial.
- Generar una **fuente analítica estructurada** que sirva como insumo para:
  - ejercicios de visualización y business intelligence,
  - análisis de comercio exterior y desempeño exportador,
  - futuros entrenamientos de modelos estadísticos o de machine learning.

Este notebook **no busca reemplazar ni reinterpretar** la metodología oficial del DANE, sino **facilitar el acceso analítico** a la información, respetando las definiciones, clasificaciones y alcances establecidos en la documentación técnica de la operación estadística de exportaciones.

## Documentación de referencia

Para el detalle metodológico, definición de variables, clasificaciones, reglas de validación y alcance estadístico, se recomienda consultar la documentación oficial del DANE:

- *Colombia – Estadísticas de Exportaciones (EXPO)*  
  Dirección de Metodología y Producción Estadística (DIMPE), DANE :contentReference[oaicite:1]{index=1}

Dicho documento describe, entre otros aspectos:
- la estructura y contenido de las declaraciones de exportación (DEX),
- las variables de clasificación y análisis (país de destino, subpartida arancelaria, valor FOB, pesos, modalidad, etc.),
- los procesos de validación, consolidación y control de calidad aplicados por el DANE,
- y el marco conceptual bajo el cual se producen las etadísticas oficiales de comercio exterior.


## Estructura del repo

```
data-analytics-portfolio-nicolas-yepes/
├── README.md                  # landin)
├── projects/
│   └── dian-export-etl/
│       ├── README.md          # README del proyecto
│       ├── notebooks/
│       │   └── ETL_Master_file_github.ipynb
│       ├── src/
│       │   ├── dian_export_sync.py
│       │   ├── dian_extract_zip.py
│       │   ├── convert_xlsx_to_csv.py
│       │   └── consolidar_csv.py
│       ├── data/              # ignorado por git
│       │   └── .gitkeep
│       ├── logs/              # ignorado por git
│       │   └── .gitkeep
│       ├── requirements.txt
│       └── .gitignore

```

## Cómo ejecutar

1. Crear entorno e instalar dependencias:
   - `pip install -r requirements.txt`
2. Asegurar que los módulos propios estén en `src/`.
3. Ejecutar el notebook en orden.


In [1]:
# ------------------------------------------------------------
# Configuración de rutas y entorno del proyecto
# ------------------------------------------------------------
# Este bloque permite que el notebook sea reproducible y
# ejecutable desde distintos puntos del repositorio.
# ------------------------------------------------------------

from pathlib import Path
from importlib import reload
import sys
import pandas as pd

# Directorio actual
HERE = Path.cwd().resolve()

# Detección de la raíz del proyecto
if (HERE / "src").exists():
    REPO_ROOT = HERE
elif (HERE.parent / "src").exists():
    REPO_ROOT = HERE.parent
else:
    raise RuntimeError(
        "No se pudo detectar la raíz del proyecto. "
           )

# Definición de directorios
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_zip"
EXTRACTED_DIR = DATA_DIR / "extracted"
CSV_DIR = DATA_DIR / "csv"
CONSOLIDATED_DIR = DATA_DIR / "consolidated"
PROCESSED_DIR = DATA_DIR / "processed"
LOG_DIR = REPO_ROOT / "logs"

# Crear carpetas necesarias
for d in [RAW_DIR, EXTRACTED_DIR, CSV_DIR, CONSOLIDATED_DIR, PROCESSED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Outputs
OUTPUT_CSV = CONSOLIDATED_DIR / "exportaciones_consolidadas.csv"
OUTPUT_PARQUET = PROCESSED_DIR / "exportaciones_limpias.parquet"

# Añadir src/ al path
SRC_DIR = REPO_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

# Configuración pandas
pd.set_option("display.max_columns", 200)



## 1) Descarga de ZIPs (RAW)

Este paso descarga los archivos ZIP al directorio `data/raw_zip/`.  
Si ya estan descargados, saltar esta sección.


In [ ]:
# Importación dinámica del módulo de descarga DIAN/DANE
# Este bloque importa y recarga dinámicamente el módulo
# encargado de sincronizar los archivos ZIP mensuales
# de exportaciones.

try:
    # Importar el módulo completo
    import dian_export_sync

    # Forzar recarga del módulo para reflejar cambios recientes a "dian_export_sync.py"
    reload(dian_export_sync)

    # Importar la función principal de sincronización que descarga de manera automatica los archivos zip directamente de web DIAN.
    from dian_export_sync import sync_dian_exportaciones

except Exception as e:
    # Error controlado y mensaje claro si el módulo no está disponible
    raise ImportError(
        "No se puede importar 'dian_export_sync'. "
        "Revisar que el archivo exista en ./src/dian_export_sync.py "
        "y que la carpeta 'src/' esté correctamente configurada en el path."
    ) from e

# Descarga incremental de archivos ZIP de exportaciones
# La función identifica y descarga únicamente los archivos
# que no existen localmente, comenzando desde el período
# especificado (año y mes).

nuevos = sync_dian_exportaciones(
    dest_dir=RAW_DIR,    # Carpeta donde se almacenan los ZIP crudos
    desde_anio=2021,     # Año inicial del histórico
    desde_mes=1,         # Mes inicial
    verbose=True         # Logs detallados de la descarga
)

# Resumen de archivos descargados en esta ejecución

print("Nuevos ZIP descargados:")
for f in nuevos:
    # Mostrar solo el nombre del archivo (no el path completo)
    print(" -", Path(f).name if not isinstance(f, Path) else f.name)



 Sincronizando exportaciones 2021-01 → 2026-01

 Descargando 01_Exportaciones_2021_Enero.zip...
    Guardado en: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\01_Exportaciones_2021_Enero.zip

 Descargando 02_Exportaciones_2021_Febrero.zip...
    Guardado en: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\02_Exportaciones_2021_Febrero.zip

 Descargando 03_Exportaciones_2021_Marzo.zip...
    Guardado en: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\03_Exportaciones_2021_Marzo.zip

 Descargando 04_Exportaciones_2021_Abril.zip...
    Guardado en: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\04_Exportaciones_2021_Abril.zip

 Descargando 05_Exportaciones_2021_Mayo.zip...
    Guardado en: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\05_Exportaciones_2021_Mayo.zip

 Desc

## 2) Extracción de ZIPs a carpeta `extracted/`

Extrae los ZIPs descargados para dejar los XLSX/archivos de trabajo listos para conversión.


In [4]:
# Importación dinámica del módulo de extracción de ZIPs
# Este bloque importa y recarga el módulo responsable de
# descomprimir los archivos ZIP descargados previamente.
# El uso de reload() permite reflejar cambios en el código
# del módulo sin necesidad de reiniciar el kernel.

from importlib import reload

try:
    # Importar el módulo completo
    import dian_extract_zip

    # Forzar la recarga del módulo (evita usar modulos del src desactualizados)
    reload(dian_extract_zip)

    # Importar la función principal de extracción
    from dian_extract_zip import extract_all_zips

except Exception as e:
    # Error controlado con mensaje claro si el módulo no está disponible
    raise ImportError(
        "No se puede importar 'dian_extract_zip'. "
        "Revisar de que el archivo exista en ./src/dian_extract_zip.py "
        "y que la carpeta 'src/' esté correctamente configurada en el path."
    ) from e

# Extracción incremental de archivos ZIP
# Se recorren los ZIP descargados y se extraen los archivos
# contenidos (principalmente XLSX), evitando reprocesar archivos ya existentes.

nuevas = extract_all_zips(
    zip_dir=str(RAW_DIR),         # Carpeta con ZIP crudos descargados
    extract_dir=str(EXTRACTED_DIR),  # Carpeta destino de archivos extraídos
    overwrite=False,              # No sobrescribir archivos ya extraídos
    verbose=True                  # Logging detallado del proceso
)

# Resumen de la extracción

print(f"ZIPs procesados: {len(nuevas) if nuevas is not None else 'N/D'}")



 Extrayendo: 01_Exportaciones_2021_Enero.zip
   → Carpeta destino: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\extracted\01_Exportaciones_2021_Enero
   ✅ Extraído correctamente

 Extrayendo: 01_Exportaciones_2022_Enero.zip
   → Carpeta destino: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\extracted\01_Exportaciones_2022_Enero
   ✅ Extraído correctamente

 Extrayendo: 01_Exportaciones_2023_Enero.zip
   → Carpeta destino: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\extracted\01_Exportaciones_2023_Enero
   ✅ Extraído correctamente

 Extrayendo: 01_Exportaciones_2024_Enero.zip
   → Carpeta destino: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\extracted\01_Exportaciones_2024_Enero
   ✅ Extraído correctamente

 Extrayendo: 01_Exportaciones_2025_Enero.zip
   → Carpeta destino: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\di

## 3) Conversión de XLSX a CSV

Convierte los archivos extraídos a CSV en `data/csv/`.


In [8]:
# Importación dinámica del módulo de conversión XLSX → CSV
# Este bloque importa y recarga el módulo encargado de convertir
# los archivos XLSX extraídos desde los ZIP en archivos CSV
# estandarizados. Este paso permite abandonar formatos
# ofimáticos y preparar los datos para consolidación y análisis.

from importlib import reload

try:
    # Importar el módulo completo
    import convert_xlsx_to_csv

    # Forzar recarga del módulo para reflejar cambios recientes
    reload(convert_xlsx_to_csv)

    # Importar la función principal de conversión
    from convert_xlsx_to_csv import convertir_xlsx_nuevos_a_csv

except Exception as e:
    # Error controlado con mensaje claro si el módulo no está disponible
    raise ImportError(
        "No se pudo importar 'convert_xlsx_to_csv'. "
        "Revisar que el archivo exista en ./src/convert_xlsx_to_csv.py "
        "y que la carpeta 'src/' esté correctamente configurada en el path."
    ) from e

# Conversión incremental de archivos XLSX a CSV
# Se recorren los archivos XLSX extraídos y se convierten a CSV,
# aplicando una estructura homogénea que facilita la
# consolidación posterior. El proceso registra logs para
# trazabilidad y debugging.

convertir_xlsx_nuevos_a_csv(
    extracted_dir=EXTRACTED_DIR,
    csv_dir=CSV_DIR,
    log_dir=LOG_DIR,
    verbose=True
)


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.
Log de control: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\logs\xlsx_to_csv.log
Archivos ya convertidos: 0
XLSX encontrados en disco: 58
XLSX nuevos a procesar: 58

Convirtiendo 01_Exportaciones_2021_Enero.xlsx...
   CSV guardado en: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\csv\01_Exportaciones_2021_Enero.csv

Convirtiendo 01_Exportaciones_2022_Enero.xlsx...
   CSV guardado en: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\csv\01_Exportaciones_2022_Enero.csv

Convirtiendo 01_Exportaciones_2023_Enero.xlsx...
   CSV guardado en: E:\Portafolio\data-analytics-portfolio-nicolas-ye

[WindowsPath('E:/Portafolio/data-analytics-portfolio-nicolas-yepes/projects/dian-export-etl/data/csv/01_Exportaciones_2021_Enero.csv'),
 WindowsPath('E:/Portafolio/data-analytics-portfolio-nicolas-yepes/projects/dian-export-etl/data/csv/01_Exportaciones_2022_Enero.csv'),
 WindowsPath('E:/Portafolio/data-analytics-portfolio-nicolas-yepes/projects/dian-export-etl/data/csv/01_Exportaciones_2023_Enero.csv'),
 WindowsPath('E:/Portafolio/data-analytics-portfolio-nicolas-yepes/projects/dian-export-etl/data/csv/01_Exportaciones_2024_Enero.csv'),
 WindowsPath('E:/Portafolio/data-analytics-portfolio-nicolas-yepes/projects/dian-export-etl/data/csv/01_Exportaciones_2025_Enero.csv'),
 WindowsPath('E:/Portafolio/data-analytics-portfolio-nicolas-yepes/projects/dian-export-etl/data/csv/02_Exportaciones_2021_Febrero.csv'),
 WindowsPath('E:/Portafolio/data-analytics-portfolio-nicolas-yepes/projects/dian-export-etl/data/csv/02_Exportaciones_2022_Febrero.csv'),
 WindowsPath('E:/Portafolio/data-analytics-p

## 4) Consolidación de CSVs

Une todos los CSV generados en un único archivo en `data/consolidated/`.


In [4]:
# Importación dinámica del módulo de consolidación de CSV
# Este bloque carga el módulo que consolida los CSV mensuales en un único archivo histórico (OUTPUT_CSV)

try:
    # Importar el módulo completo (para poder recargarlo sin reiniciar kernel)
    import consolidar_csv
    reload(consolidar_csv)

    # Importar la función pública principal del módulo
    from consolidar_csv import consolidar_csv_nuevos

except Exception as e:
    raise ImportError(
        "No pude importar 'consolidar_csv'. "
        "Revisar tener el archivo en ./src/consolidar_csv.py"
    ) from e


consolidar_csv_nuevos(
    csv_dir=CSV_DIR,          # Carpeta que contiene los CSV intermedios (provenientes de XLSX)
    output_csv=OUTPUT_CSV,    # Archivo de salida consolidado (unión de períodos)
    log_file=None,
    verbose=True,
)

# Confirmación del artefacto generado
print("Consolidado creado en:", OUTPUT_CSV)

Carpeta CSV: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\csv
CSV maestro: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\consolidated\exportaciones_consolidadas.csv
Log de procesados: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\consolidated\exportaciones_consolidadas_log.txt
CSV ya registrados en log: 0
CSV encontrados en disco: 58
CSV nuevos a procesar: 58
Leyendo: 01_Exportaciones_2021_Enero.csv
Leyendo: 01_Exportaciones_2022_Enero.csv
Leyendo: 01_Exportaciones_2023_Enero.csv
Leyendo: 01_Exportaciones_2024_Enero.csv
Leyendo: 01_Exportaciones_2025_Enero.csv
Leyendo: 02_Exportaciones_2021_Febrero.csv
Leyendo: 02_Exportaciones_2022_Febrero.csv
Leyendo: 02_Exportaciones_2023_Febrero.csv
Leyendo: 02_Exportaciones_2024_Febrero.csv
Leyendo: 02_Exportaciones_2025_Febrero.csv
Leyendo: 03_Exportaciones_2021_Marzo.csv
Leyendo: 03_Exportaciones_2022_Marzo.csv
Leyendo: 03_Exportacione

## 5) Carga del consolidado + revisión rápida

En este paso se realiza la carga controlada del archivo consolidado en CSV, utilizando lectura por chunks para manejar volúmenes grandes sin comprometer la memoria. Durante la lectura se hace una revisión rápida de sanidad del dataset (estructura, tamaño y columnas relevantes) y, de forma inmediata, se convierte el archivo a formato Parquet.

Se opta por un formato parquet sobre CSV porque eeduce significativamente el tamaño en disco y mejora los tiempos de lectura en análisis posteriores.

In [4]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PARQUET_OUT = OUTPUT_CSV.with_suffix(".parquet")

chunk_size = 200_000
parquet_writer = None

for i, chunk in enumerate(
    pd.read_csv(
        OUTPUT_CSV,
        sep=";",                
        engine="python",
        on_bad_lines="skip",
        dtype_backend="numpy_nullable",
        chunksize=chunk_size,
        encoding="utf-8",       
    )
):
    #quitar columnas basura tipo Unnamed
    chunk = chunk.loc[:, ~chunk.columns.astype(str).str.startswith("Unnamed")]

    table = pa.Table.from_pandas(chunk, preserve_index=False)

    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(PARQUET_OUT, table.schema, compression="snappy")

    parquet_writer.write_table(table)
    print(f"Chunk {i+1} escrito ({len(chunk):,} filas)")

parquet_writer.close()
print("Conversión a Parquet finalizada")



✔ Chunk 1 escrito (200,000 filas)
✔ Chunk 2 escrito (200,000 filas)
✔ Chunk 3 escrito (200,000 filas)
✔ Chunk 4 escrito (200,000 filas)
✔ Chunk 5 escrito (200,000 filas)
✔ Chunk 6 escrito (200,000 filas)
✔ Chunk 7 escrito (200,000 filas)
✔ Chunk 8 escrito (200,000 filas)
✔ Chunk 9 escrito (200,000 filas)
✔ Chunk 10 escrito (200,000 filas)
✔ Chunk 11 escrito (200,000 filas)
✔ Chunk 12 escrito (200,000 filas)
✔ Chunk 13 escrito (200,000 filas)
✔ Chunk 14 escrito (200,000 filas)
✔ Chunk 15 escrito (200,000 filas)
✔ Chunk 16 escrito (200,000 filas)
✔ Chunk 17 escrito (200,000 filas)
✔ Chunk 18 escrito (200,000 filas)
✔ Chunk 19 escrito (200,000 filas)
✔ Chunk 20 escrito (200,000 filas)
✔ Chunk 21 escrito (200,000 filas)
✔ Chunk 22 escrito (200,000 filas)
✔ Chunk 23 escrito (200,000 filas)
✔ Chunk 24 escrito (200,000 filas)
✔ Chunk 25 escrito (200,000 filas)
✔ Chunk 26 escrito (200,000 filas)
✔ Chunk 27 escrito (200,000 filas)
✔ Chunk 28 escrito (200,000 filas)
✔ Chunk 29 escrito (200,000 f

In [5]:
bd = pd.read_parquet(PARQUET_OUT)

bd.head()


,FECHA_PROCESO,NUMERO_SERIE,OFICINA,COD_ADUANA_DESPACHO,ADUANA_DESPACHO,TIPO_IDENT,NIT_EXPORTADOR,TIPO_USUARIO,COD_USUARIO,CLASE_EXPORTADOR,COD_DPTO_EXPORTADOR,COD_PAIS_DESTINO_NUM,COD_PAIS_DESTINO_ALF,COD_PAIS_DESTINO,PAIS_DESTINO_FINAL,CIUDAD_DESTINATARIO,NUM_SOLICITUD_AUTO_EMBARQUE,TIPO_DECLARACION,TIPO_DESPACHO,COD_LUGAR_SALIDA_NUM,COD_LUG_SALIDA_ALF,COD_REGION_PROCEDENCIA,REGION_PROCEDENCIA,NUM__DECLA_EXPORTACION_ANT,FECH_DECLA_EXPORTACION_ANT,NUM_DECLARACION_PRECEDENTE,FECH_DECLA_PRECEDENTE,COD_MODALIDAD_PRECEDENTE,COD_MONEDA_TRANSACCION,COD_MODO_TRANSPORTE,MODO_TRANSPORTE,BANDERA,COD_NACIONALIDAD_BANDERA,NACIONALIDAD_BANDERA,COD_REGIMEN_CAN,COD_MODALIDAD_EXPORTACION,MODALIDAD_EXPORTACION,FORMA_PAGO,COD_TIPO_EMBARQUE,TIPO_DE_EMBARQUE,COD_TIPO_DATOS,TIPO_DE_DATOS,TIPO_CERTIFICADO_ORIGEN,SISTEMAS_ESPECIALES,COD_EXPORTACION_TRANSITO,EXPORTACION_EN_TRANSITO,SUBPARTIDA,COD_REGION_ORIGEN,REGION_DE_ORIGEN,COD_UNIDAD_FISICA_NUM,COD_UNIDAD_FISICA_ALF,UNIDAD_FISICA,CANTIDAD_UNIDADES_FISICAS,PESO_BRUTO_KGS,PESO_NETO_KGS,VALOR_FOB_USD,VALOR_FOB_PESOS,VLR_SERIE_AGREGADO_NAL_USD,VALOR_SERIE_FLETES_USD,VALOR_SERIE_SEGUROS_USD,VLR_SERIE_OTROS_GASTOS_USD,COD_ADUANA_SALIDA,ADUANA_SALIDA,FECHA_SOLICITUD_AUTO_EMBARQUE,NUMERO_FORMULARIO,FECHA_DECLARACION_EXPORTACION,RAZON_SOCIAL_EXPORTADOR,DIREC_EXPORTADOR,NIT_DECLARANTE,RAZON_SOCIAL_DECLARANTE,RAZON_SOCIAL_DESTINATARIO,DOMICILIO_DESTINATARIO,origen_archivo
0,202101,1,99,39,Impuestos y Aduanas de Maicao,2,890204396.0,36,0,2,44430,850,VEN,VE,VENEZUELA,PUNTA CARDON,6027672357779.0,1,Inicial,39,MAI,47,MAGDALENA,0,0,0,0,<NA>,USD,3,Terrestre (carretero),13,AF,AFGANISTAN,1,198,Exportacion definitiva de mercancias de fabric...,1,1,Unico,1,Definitivos al embarque,8,NO,N,NO,1101000000,47,MAGDALENA,33,KG,Kilogramo,32000.0,32080.0,32000.0,13888.0,48958672.0,0.0,0.0,0.0,0.0,39,Impuestos y Aduanas de Maicao,202101,6007671773173,20210125,MOLINOS SAN MIGUEL S.A.S,CL 14 10 111 OF 02 BRR CENTRO,900117029.0,AGENCIA DE ADUANAS MUNDIAL DE SERVICIOS INTERN...,"HOTEL LA PUERTA INN, C.A.",AV GENERAL PELAYO E/MANPORAL Y TOSTO LOCAL S7N...,01_Exportaciones_2021_Enero.csv
1,202101,1,99,48,Aduanas de Cartagena,2,890904138.0,36,0,2,5001,105,BRA,BR,BRASIL,MANAUS,6027667078185.0,1,Inicial,48,CTG,5,ANTIOQUIA,0,0,0,0,<NA>,USD,1,Marítimo,23,DE,ALEMANIA,1,198,Exportacion definitiva de mercancias de fabric...,1,1,Unico,1,Definitivos al embarque,8,SI,N,NO,3912310000,5,ANTIOQUIA,33,KG,Kilogramo,5000.0,5140.0,5000.0,26215.0,90461149.1,20230.84,0.0,0.0,0.0,48,Aduanas de Cartagena,202101,6007671438415,20210106,AMTEX S.A.,CL 8 B CR 65 191 IN 511 ED PUERTO SECO,890902266.0,AGENCIA DE ADUANAS MARIO LONDOÑO S.A. NIVEL 1,RECOFARMA INDUSTRIA DO AMAZONAS LTDA.,"AV.BURITI, 190 DISTRITO INDUSTRIAL 69075-000 M...",01_Exportaciones_2021_Enero.csv
2,202101,4,99,48,Aduanas de Cartagena,2,890904138.0,36,0,2,5001,105,BRA,BR,BRASIL,SAO PAULO,6027667084021.0,1,Inicial,48,CTG,5,ANTIOQUIA,0,0,0,0,<NA>,USD,1,Marítimo,23,DE,ALEMANIA,1,198,Exportacion definitiva de mercancias de fabric...,1,1,Unico,1,Definitivos al embarque,8,SI,N,NO,3912310000,5,ANTIOQUIA,33,KG,Kilogramo,5000.0,5120.0,5000.0,16390.0,56995897.2,10406.05,184.48,16.3,0.0,48,Aduanas de Cartagena,202101,6007672165501,20210122,AMTEX S.A.,CL 8 B CR 65 191 IN 511 ED PUERTO SECO,890902266.0,AGENCIA DE ADUANAS MARIO LONDOÑO S.A. NIVEL 1,QUIMICAMTEX LTDA,"RUA HENRIQUE SCHAUMANN No. 286, CONJUNTO 64 ED...",01_Exportaciones_2021_Enero.csv
3,202101,1,99,48,Aduanas de Cartagena,2,890904138.0,36,0,2,5001,105,BRA,BR,BRASIL,MANAUS,6027667078146.0,1,Inicial,48,CTG,5,ANTIOQUIA,0,0,0,0,<NA>,USD,1,Marítimo,23,DE,ALEMANIA,1,198,Exportacion definitiva de mercancias de fabric...,1,1,Unico,1,Definitivos al embarque,8,SI,N,NO,3912310000,5,ANTIOQUIA,33,KG,Kilogramo,15000.0,15420.0,15000.0,76125.0,262687582.5,58172.52,0.0,0.0,0.0,48,Aduanas de Cartagena,202101,6007671438408,20210106,AMTEX S.A.,CL 8 B CR 65 191 IN 511 ED PUERTO SECO,890902266.0,AGENCIA DE ADUANAS MARIO LONDOÑO S.A. NIVEL 1,RECOFARMA INDUSTRIA DO A

In [10]:
print(list(bd.columns))
print(bd.shape)

['FECHA_PROCESO', 'NUMERO_SERIE', 'OFICINA', 'COD_ADUANA_DESPACHO', 'ADUANA_DESPACHO', 'TIPO_IDENT', 'NIT_EXPORTADOR', 'TIPO_USUARIO', 'COD_USUARIO', 'CLASE_EXPORTADOR', 'COD_DPTO_EXPORTADOR', 'COD_PAIS_DESTINO_NUM', 'COD_PAIS_DESTINO_ALF', 'COD_PAIS_DESTINO', 'PAIS_DESTINO_FINAL', 'CIUDAD_DESTINATARIO', 'NUM_SOLICITUD_AUTO_EMBARQUE', 'TIPO_DECLARACION', 'TIPO_DESPACHO', 'COD_LUGAR_SALIDA_NUM', 'COD_LUG_SALIDA_ALF', 'COD_REGION_PROCEDENCIA', 'REGION_PROCEDENCIA', 'NUM__DECLA_EXPORTACION_ANT', 'FECH_DECLA_EXPORTACION_ANT', 'NUM_DECLARACION_PRECEDENTE', 'FECH_DECLA_PRECEDENTE', 'COD_MODALIDAD_PRECEDENTE', 'COD_MONEDA_TRANSACCION', 'COD_MODO_TRANSPORTE', 'MODO_TRANSPORTE', 'BANDERA', 'COD_NACIONALIDAD_BANDERA', 'NACIONALIDAD_BANDERA', 'COD_REGIMEN_CAN', 'COD_MODALIDAD_EXPORTACION', 'MODALIDAD_EXPORTACION', 'FORMA_PAGO', 'COD_TIPO_EMBARQUE', 'TIPO_DE_EMBARQUE', 'COD_TIPO_DATOS', 'TIPO_DE_DATOS', 'TIPO_CERTIFICADO_ORIGEN', 'SISTEMAS_ESPECIALES', 'COD_EXPORTACION_TRANSITO', 'EXPORTACION_EN_T

## 6) Limpieza inicial: nulos y duplicados

- Remoción de filas completamente nulas (si aplica).
- Detección y eliminación de duplicados.


In [13]:
BD_SIN_NULL = bd.dropna(how="all")
BD_SIN_DUP = BD_SIN_NULL.drop_duplicates()
BD_SIN_DUP.shape

(5947458, 73)

## 7) Tipos de datos y preparación (especialmente fechas y códigos)

Aquí revisamos dtypes y estandarizamos:
- Fechas a `datetime64[ns]` cuando aplique.
- Códigos/IDs a `string` (para no confundir categorias con valores númericos).
- Asegurar el formato de los valores númericos


In [14]:
pd.set_option("display.max_rows", None)

BD_SIN_DUP.dtypes.reset_index(name="dtype").rename(columns={"index": "columna"})

,columna,dtype
0,FECHA_PROCESO,Int64
1,NUMERO_SERIE,Int64
2,OFICINA,Int64
3,COD_ADUANA_DESPACHO,Int64
4,ADUANA_DESPACHO,string[python]
5,TIPO_IDENT,Int64
6,NIT_EXPORTADOR,Float64
7,TIPO_USUARIO,Int64
8,COD_USUARIO,Int64
9,CLASE_EXPORTADOR,Int64


In [15]:
# Muestra del contenido de columnas de fecha (útil para detectar formatos int/str)
cols_fecha = [
    "FECHA_PROCESO",
    "FECH_DECLA_EXPORTACION_ANT",
    "FECH_DECLA_PRECEDENTE",
    "FECHA_SOLICITUD_AUTO_EMBARQUE",
    "FECHA_DECLARACION_EXPORTACION",
]

cols_fecha_presentes = [c for c in cols_fecha if c in BD_SIN_DUP.columns]
BD_SIN_DUP[cols_fecha_presentes].head()

,FECHA_PROCESO,FECH_DECLA_EXPORTACION_ANT,FECH_DECLA_PRECEDENTE,FECHA_SOLICITUD_AUTO_EMBARQUE,FECHA_DECLARACION_EXPORTACION
0,202101,0,0,202101,20210125
1,202101,0,0,202101,20210106
2,202101,0,0,202101,20210122
3,202101,0,0,202101,20210106
4,202101,0,0,202101,20210122


In [23]:
#Selección de fecha del la BD y ajuste de formato.
#Teniendo en cuenta que FECHA_DECLARACION_EXPORTACION es la fecha oficial y más granular en cada una de las operaciones de exportación
#se toma esta como la variable fecha guía y se eliminar el resto de columnas
BD_FORM_CORR = (
    BD_SIN_DUP
    .drop(columns=[
        "FECHA_PROCESO",
        "FECH_DECLA_EXPORTACION_ANT",
        "FECH_DECLA_PRECEDENTE",
        "FECHA_SOLICITUD_AUTO_EMBARQUE"
    ])
    .assign(
        FECHA_DECLARACION_EXPORTACION=lambda df: pd.to_datetime(
            df["FECHA_DECLARACION_EXPORTACION"]
            .replace(0, pd.NA)
            .astype(str),
            format="%Y%m%d",
            errors="coerce"
        )
    )
)

In [24]:
#Ajuste de formato de variables codigo que no tienen un valor númerico real

cols_to_string = [
    "NUMERO_SERIE",
    "OFICINA",
    "COD_ADUANA_DESPACHO",
    "TIPO_IDENT",
    "TIPO_USUARIO",
    "COD_USUARIO",
    "CLASE_EXPORTADOR",
    "COD_DPTO_EXPORTADOR",
    "COD_PAIS_DESTINO_NUM",
    "COD_LUGAR_SALIDA_NUM",
    "COD_REGION_PROCEDENCIA",
    "NUM__DECLA_EXPORTACION_ANT",
    "NUM_DECLARACION_PRECEDENTE",
    "COD_MODO_TRANSPORTE",
    "NIT_EXPORTADOR",
    "NIT_DECLARANTE",
    "BANDERA",
    "COD_REGIMEN_CAN",
    "COD_MODALIDAD_EXPORTACION",
    "NUM_SOLICITUD_AUTO_EMBARQUE",
    "FORMA_PAGO",
    "COD_TIPO_EMBARQUE",
    "COD_TIPO_DATOS",
    "TIPO_CERTIFICADO_ORIGEN",
    "SUBPARTIDA",
    "COD_REGION_ORIGEN",
    "COD_UNIDAD_FISICA_NUM",
    "COD_ADUANA_SALIDA",
    "TIPO_DECLARACION",
    "NUMERO_FORMULARIO",
]

BD_FORM_CORR[cols_to_string] = (
    BD_FORM_CORR[cols_to_string]
    .astype("string")
)

In [25]:
#Ajuste del formato de valores númericos
pd.options.display.float_format = '{:,.2f}'.format

BD_FORM_CORR[
    [
        "CANTIDAD_UNIDADES_FISICAS",
        "PESO_BRUTO_KGS",
        "PESO_NETO_KGS",
        "VALOR_FOB_USD",
        "VALOR_FOB_PESOS",
        "VLR_SERIE_AGREGADO_NAL_USD",
        "VALOR_SERIE_FLETES_USD",
        "VALOR_SERIE_SEGUROS_USD",
        "VLR_SERIE_OTROS_GASTOS_USD",
    ]
].head()

,CANTIDAD_UNIDADES_FISICAS,PESO_BRUTO_KGS,PESO_NETO_KGS,VALOR_FOB_USD,VALOR_FOB_PESOS,VLR_SERIE_AGREGADO_NAL_USD,VALOR_SERIE_FLETES_USD,VALOR_SERIE_SEGUROS_USD,VLR_SERIE_OTROS_GASTOS_USD
0,"32,000.00","32,080.00","32,000.00","13,888.00","48,958,672.00",0.00,0.00,0.00,0.00
1,"5,000.00","5,140.00","5,000.00","26,215.00","90,461,149.10","20,230.84",0.00,0.00,0.00
2,"5,000.00","5,120.00","5,000.00","16,390.00","56,995,897.20","10,406.05",184.48,16.30,0.00
3,"15,000.00","15,420.00","15,000.00","76,125.00","262,687,582.50","58,172.52",0.00,0.00,0.00
4,"3,000.00","3,072.00","3,000.00","8,942.00","31,095,626.16","5,351.63",100.65,8.90,0.00


## 8) Export final (para análisis / modelos)

Guardamos el dataset limpio en **Parquet** (más eficiente que CSV) y opcionalmente en CSV.


In [26]:
print("Dimensión:", BD_FORM_CORR.shape)
display(BD_FORM_CORR.head(5))

Dimensión: (5947458, 69)


,NUMERO_SERIE,OFICINA,COD_ADUANA_DESPACHO,ADUANA_DESPACHO,TIPO_IDENT,NIT_EXPORTADOR,TIPO_USUARIO,COD_USUARIO,CLASE_EXPORTADOR,COD_DPTO_EXPORTADOR,COD_PAIS_DESTINO_NUM,COD_PAIS_DESTINO_ALF,COD_PAIS_DESTINO,PAIS_DESTINO_FINAL,CIUDAD_DESTINATARIO,NUM_SOLICITUD_AUTO_EMBARQUE,TIPO_DECLARACION,TIPO_DESPACHO,COD_LUGAR_SALIDA_NUM,COD_LUG_SALIDA_ALF,COD_REGION_PROCEDENCIA,REGION_PROCEDENCIA,NUM__DECLA_EXPORTACION_ANT,NUM_DECLARACION_PRECEDENTE,COD_MODALIDAD_PRECEDENTE,COD_MONEDA_TRANSACCION,COD_MODO_TRANSPORTE,MODO_TRANSPORTE,BANDERA,COD_NACIONALIDAD_BANDERA,NACIONALIDAD_BANDERA,COD_REGIMEN_CAN,COD_MODALIDAD_EXPORTACION,MODALIDAD_EXPORTACION,FORMA_PAGO,COD_TIPO_EMBARQUE,TIPO_DE_EMBARQUE,COD_TIPO_DATOS,TIPO_DE_DATOS,TIPO_CERTIFICADO_ORIGEN,SISTEMAS_ESPECIALES,COD_EXPORTACION_TRANSITO,EXPORTACION_EN_TRANSITO,SUBPARTIDA,COD_REGION_ORIGEN,REGION_DE_ORIGEN,COD_UNIDAD_FISICA_NUM,COD_UNIDAD_FISICA_ALF,UNIDAD_FISICA,CANTIDAD_UNIDADES_FISICAS,PESO_BRUTO_KGS,PESO_NETO_KGS,VALOR_FOB_USD,VALOR_FOB_PESOS,VLR_SERIE_AGREGADO_NAL_USD,VALOR_SERIE_FLETES_USD,VALOR_SERIE_SEGUROS_USD,VLR_SERIE_OTROS_GASTOS_USD,COD_ADUANA_SALIDA,ADUANA_SALIDA,NUMERO_FORMULARIO,FECHA_DECLARACION_EXPORTACION,RAZON_SOCIAL_EXPORTADOR,DIREC_EXPORTADOR,NIT_DECLARANTE,RAZON_SOCIAL_DECLARANTE,RAZON_SOCIAL_DESTINATARIO,DOMICILIO_DESTINATARIO,origen_archivo
0,1,99,39,Impuestos y Aduanas de Maicao,2,890204396.0,36,0,2,44430,850,VEN,VE,VENEZUELA,PUNTA CARDON,6027672357779.0,1,Inicial,39,MAI,47,MAGDALENA,0,0,<NA>,USD,3,Terrestre (carretero),13,AF,AFGANISTAN,1,198,Exportacion definitiva de mercancias de fabric...,1,1,Unico,1,Definitivos al embarque,8,NO,N,NO,1101000000,47,MAGDALENA,33,KG,Kilogramo,"32,000.00","32,080.00","32,000.00","13,888.00","48,958,672.00",0.00,0.00,0.00,0.00,39,Impuestos y Aduanas de Maicao,6007671773173,2021-01-25,MOLINOS SAN MIGUEL S.A.S,CL 14 10 111 OF 02 BRR CENTRO,900117029.0,AGENCIA DE ADUANAS MUNDIAL DE SERVICIOS INTERN...,"HOTEL LA PUERTA INN, C.A.",AV GENERAL PELAYO E/MANPORAL Y TOSTO LOCAL S7N...,01_Exportaciones_2021_Enero.csv
1,1,99,48,Aduanas de Cartagena,2,890904138.0,36,0,2,5001,105,BRA,BR,BRASIL,MANAUS,6027667078185.0,1,Inicial,48,CTG,5,ANTIOQUIA,0,0,<NA>,USD,1,Marítimo,23,DE,ALEMANIA,1,198,Exportacion definitiva de mercancias de fabric...,1,1,Unico,1,Definitivos al embarque,8,SI,N,NO,3912310000,5,ANTIOQUIA,33,KG,Kilogramo,"5,000.00","5,140.00","5,000.00","26,215.00","90,461,149.10","20,230.84",0.00,0.00,0.00,48,Aduanas de Cartagena,6007671438415,2021-01-06,AMTEX S.A.,CL 8 B CR 65 191 IN 511 ED PUERTO SECO,890902266.0,AGENCIA DE ADUANAS MARIO LONDOÑO S.A. NIVEL 1,RECOFARMA INDUSTRIA DO AMAZONAS LTDA.,"AV.BURITI, 190 DISTRITO INDUSTRIAL 69075-000 M...",01_Exportaciones_2021_Enero.csv
2,4,99,48,Aduanas de Cartagena,2,890904138.0,36,0,2,5001,105,BRA,BR,BRASIL,SAO PAULO,6027667084021.0,1,Inicial,48,CTG,5,ANTIOQUIA,0,0,<NA>,USD,1,Marítimo,23,DE,ALEMANIA,1,198,Exportacion definitiva de mercancias de fabric...,1,1,Unico,1,Definitivos al embarque,8,SI,N,NO,3912310000,5,ANTIOQUIA,33,KG,Kilogramo,"5,000.00","5,120.00","5,000.00","16,390.00","56,995,897.20","10,406.05",184.48,16.30,0.00,48,Aduanas de Cartagena,6007672165501,2021-01-22,AMTEX S.A.,CL 8 B CR 65 191 IN 511 ED PUERTO SECO,890902266.0,AGENCIA DE ADUANAS MARIO LONDOÑO S.A. NIVEL 1,QUIMICAMTEX LTDA,"RUA HENRIQUE SCHAUMANN No. 286, CONJUNTO 64 ED...",01_Exportaciones_2021_Enero.csv
3,1,99,48,Aduanas de Cartagena,2,890904138.0,36,0,2,5001,105,BRA,BR,BRASIL,MANAUS,6027667078146.0,1,Inicial,48,CTG,5,ANTIOQUIA,0,0,<NA>,USD,1,Marítimo,23,DE,ALEMANIA,1,198,Exportacion definitiva de mercancias de fabric...,1,1,Unico,1,Definitivos al embarque,8,SI,N,NO,3912310000,5,ANTIOQUIA,33,KG,Kilogramo,"15,000.00","15,420.00","15,000.00","76,125.00","262,687,582.50","58,172.52",0.00,0.00,0.00,48,Aduanas de Cartagena,6007671438408,2021-01-06,AMTEX S.A.,CL 8 B CR 65 191 IN 511 ED PUERTO SECO,890902266.0,AGENCIA DE ADUANAS MARIO LONDOÑO S.A. NIVEL 1,RECOFARMA INDUSTRIA DO AMAZONAS LTDA.,"AV.BURITI, 190 DISTRITO INDUSTRIAL 69

In [27]:
# Describe para cuantitativas
cols_num = BD_FORM_CORR.select_dtypes(include=["number"]).columns

display(
    BD_FORM_CORR[cols_num].describe()
)

,CANTIDAD_UNIDADES_FISICAS,PESO_BRUTO_KGS,PESO_NETO_KGS,VALOR_FOB_USD,VALOR_FOB_PESOS,VLR_SERIE_AGREGADO_NAL_USD,VALOR_SERIE_FLETES_USD,VALOR_SERIE_SEGUROS_USD,VLR_SERIE_OTROS_GASTOS_USD
count,"5,947,458.00","5,947,458.00","5,947,458.00","5,947,458.00","5,947,458.00","5,947,458.00","5,947,458.00","5,947,458.00","5,947,458.00"
mean,"80,169.78","82,204.77","81,831.88","40,168.36","165,418,460.86","10,123.49",777.74,9.66,30.28
std,"3,377,528.30","2,458,120.23","2,457,871.72","822,911.39","3,397,501,323.46","248,913.83","466,027.93","6,797.77","5,113.70"
min,0.01,0.00,0.00,0.01,0.00,0.00,0.00,0.00,0.00
25%,43.44,29.30,25.14,376.64,"1,540,801.24",0.00,0.00,0.00,0.00
50%,420.00,251.19,220.00,"2,008.80","8,230,029.44",0.00,0.00,0.00,0.00
75%,"4,730.40","2,334.99","2,100.00","10,851.45","44,461,081.42",626.37,3.11,0.00,0.00
max,"3,924,109,000.00","286,695,276.00","286,695,276.00","163,250,746.99","678,744,499,523.38","76,086,959.85","802,559,109.38","16,490,670.00","6,780,716.64"


In [28]:
# Guardar Parquet final
BD_FORM_CORR.to_parquet(OUTPUT_PARQUET, index=False)
print("Dataset limpio guardado en:", OUTPUT_PARQUET)

Dataset limpio guardado en: C:\Users\nyepes\DIAN\data\processed\exportaciones_limpias.parquet
